In [1]:
import slangpy as spy
import pathlib
import matplotlib.pyplot as plt
import numpy as np
import jax
from jax import grad
import jax.numpy as jnp

In [2]:
device = spy.create_device(
    include_paths=[
        pathlib.Path(".").absolute()
    ]
)

[INFO] (rhi) layer: CreateDevice: Debug layer is enabled.
[WARN] No supported shader model found, pretending to support sm_6_0.


In [3]:
program = device.load_program("slang/auto-diff.slang", entry_point_names=["computeCov"])
kernel = device.create_compute_kernel(program)
kernel

ComputeKernel(0x6000028757c0)

In [4]:
SIZE=8

In [5]:
gaussian_buf = device.create_buffer(
    element_count=SIZE,
    struct_type=kernel.reflection.gaussians,
    usage=spy.BufferUsage.shader_resource
)
gaussian_buf

Buffer(
  device = 0x1189e9cb8,
  size = 128,
  struct_size = 16,
  format = undefined,
  usage = shader_resource,
  memory_type = device_local,
  memory_usage = 128 B,
  label = 
)

In [6]:
gaussian_cursor = spy.BufferCursor(
    kernel.reflection.gaussians.type_layout.element_type_layout,
    gaussian_buf
)
gaussian_cursor

Object(0x11dc07b28)

In [7]:
for i in range(len(gaussian_cursor)):
    gaussian_cursor[i].write({
        "rotation": float(i) / SIZE * np.pi * 2,
        "scale": np.random.rand(2).astype(np.float32)
    })
gaussian_cursor.apply()

In [8]:
for i in range(len(gaussian_cursor)):
    print(gaussian_cursor[i])

{'rotation': 0.0, 'scale': {0.63823426, 0.76152444}} [Gaussian]
{'rotation': 0.7853981852531433, 'scale': {0.12783036, 0.24192446}} [Gaussian]
{'rotation': 1.5707963705062866, 'scale': {0.51951957, 0.9237866}} [Gaussian]
{'rotation': 2.356194496154785, 'scale': {0.41938034, 0.80288166}} [Gaussian]
{'rotation': 3.1415927410125732, 'scale': {0.40137684, 0.9961457}} [Gaussian]
{'rotation': 3.9269907474517822, 'scale': {0.37629184, 0.7955093}} [Gaussian]
{'rotation': 4.71238899230957, 'scale': {0.59479946, 0.12614499}} [Gaussian]
{'rotation': 5.497786998748779, 'scale': {0.63577825, 0.816429}} [Gaussian]


In [9]:
cov_buf = device.create_buffer(
    element_count=SIZE,
    struct_type=kernel.reflection.covariance,
    usage=spy.BufferUsage.unordered_access
)

In [10]:
cov_cursor = spy.BufferCursor(
    kernel.reflection.covariance.type_layout.element_type_layout,
    cov_buf
)
for i in range(len(cov_cursor)):
    print(cov_cursor[i])

{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]


In [11]:
kernel.dispatch(
    thread_count=[SIZE, 1, 1],
    vars={
        "gaussians": gaussian_buf,
        "covariance": cov_buf
    }
)

In [12]:
cov_cursor = spy.BufferCursor(
    kernel.reflection.covariance.type_layout.element_type_layout,
    cov_buf
)
for i in range(len(cov_cursor)):
    print(cov_cursor[i])

{{0.40734297, 0}, {0, 0.57991946}} [matrix<float,2,2>]
{{0.0081703, 0}, {0, 0.029263724}} [matrix<float,2,2>]
{{5.156951e-16, 0}, {0, 1.6305438e-15}} [matrix<float,2,2>]
{{0.08793993, 0}, {0, 0.32230946}} [matrix<float,2,2>]
{{0.16110337, 0}, {0, 0.9923063}} [matrix<float,2,2>]
{{0.070797786, 0}, {0, 0.31641752}} [matrix<float,2,2>]
{{5.0309416e-17, 0}, {0, 2.2628105e-18}} [matrix<float,2,2>]
{{0.20210695, 0}, {0, 0.33327812}} [matrix<float,2,2>]


In [13]:
cov_arr = jnp.array(cov_buf.to_numpy().view(np.float32))
cov_arr

Array([4.0734297e-01, 0.0000000e+00, 0.0000000e+00, 5.7991946e-01,
       8.1703002e-03, 0.0000000e+00, 0.0000000e+00, 2.9263724e-02,
       5.1569510e-16, 0.0000000e+00, 0.0000000e+00, 1.6305438e-15,
       8.7939933e-02, 0.0000000e+00, 0.0000000e+00, 3.2230946e-01,
       1.6110337e-01, 0.0000000e+00, 0.0000000e+00, 9.9230629e-01,
       7.0797786e-02, 0.0000000e+00, 0.0000000e+00, 3.1641752e-01,
       5.0309416e-17, 0.0000000e+00, 0.0000000e+00, 2.2628105e-18,
       2.0210695e-01, 0.0000000e+00, 0.0000000e+00, 3.3327812e-01],      dtype=float32)

In [14]:
mean, grad = jax.value_and_grad(jnp.mean)(cov_arr)
mean, grad

(Array(0.10971737, dtype=float32),
 Array([0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125,
        0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125,
        0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125,
        0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125,
        0.03125, 0.03125, 0.03125, 0.03125], dtype=float32))

In [15]:
cov_diff_buf = device.create_buffer(
    element_count=8,
    struct_type=kernel.reflection.covarianceDiff,
    usage=spy.BufferUsage.shader_resource
)
cov_diff_cursor = spy.BufferCursor(
    kernel.reflection.covarianceDiff.type_layout.element_type_layout,
    cov_diff_buf
)

In [16]:
grad = grad.reshape(SIZE, 2, 2).astype(jnp.float32)
grad.shape

(8, 2, 2)

In [17]:
for i in range(len(cov_diff_cursor)):
    print(cov_diff_cursor[i])

{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]


In [18]:
for i in range(len(cov_diff_cursor)):
    cov_diff_cursor[i].write(grad[i])
cov_diff_cursor.apply()

In [19]:
for i in range(len(cov_diff_cursor)):
    print(cov_diff_cursor[i])

{{0.03125, 0.03125}, {0.03125, 0.03125}} [matrix<float,2,2>]
{{0.03125, 0.03125}, {0.03125, 0.03125}} [matrix<float,2,2>]
{{0.03125, 0.03125}, {0.03125, 0.03125}} [matrix<float,2,2>]
{{0.03125, 0.03125}, {0.03125, 0.03125}} [matrix<float,2,2>]
{{0.03125, 0.03125}, {0.03125, 0.03125}} [matrix<float,2,2>]
{{0.03125, 0.03125}, {0.03125, 0.03125}} [matrix<float,2,2>]
{{0.03125, 0.03125}, {0.03125, 0.03125}} [matrix<float,2,2>]
{{0.03125, 0.03125}, {0.03125, 0.03125}} [matrix<float,2,2>]


In [20]:
gaussian_diff_buf = device.create_buffer(
    element_count=8,
    struct_type=kernel.reflection.gaussianDiffs,
    usage=spy.BufferUsage.unordered_access
)

In [21]:
gaussian_diff_cursor = spy.BufferCursor(
    kernel.reflection.gaussianDiffs.type_layout.element_type_layout,
    gaussian_diff_buf
)
for i in range(len(gaussian_diff_cursor)):
    print(gaussian_diff_cursor[i])

{'rotation': 0.0, 'scale': {0, 0}} [Gaussian]
{'rotation': 0.0, 'scale': {0, 0}} [Gaussian]
{'rotation': 0.0, 'scale': {0, 0}} [Gaussian]
{'rotation': 0.0, 'scale': {0, 0}} [Gaussian]
{'rotation': 0.0, 'scale': {0, 0}} [Gaussian]
{'rotation': 0.0, 'scale': {0, 0}} [Gaussian]
{'rotation': 0.0, 'scale': {0, 0}} [Gaussian]
{'rotation': 0.0, 'scale': {0, 0}} [Gaussian]


In [22]:
program_bwd = device.load_program("slang/auto-diff.slang", entry_point_names=["computeCovDiff"])
kernel_bwd = device.create_compute_kernel(program_bwd)
kernel_bwd

ComputeKernel(0x600002892f80)

In [23]:
kernel_bwd.dispatch(
    thread_count=[SIZE, 1, 1],
    vars={
        "gaussians": gaussian_buf,
        "gaussianDiffs": gaussian_diff_buf,
        "covarianceDiff": cov_diff_buf
    }
)

In [24]:
gaussian_diff_cursor = spy.BufferCursor(
    kernel.reflection.gaussianDiffs.type_layout.element_type_layout,
    gaussian_diff_buf
)
for i in range(len(gaussian_diff_cursor)):
    print(gaussian_diff_cursor[i])

{'rotation': 0.0, 'scale': {0.03988964, 0.047595277}} [Gaussian]
{'rotation': -0.0023396264296025038, 'scale': {0.0039946986, 0.0075601395}} [Gaussian]
{'rotation': 3.0687639096527164e-09, 'scale': {6.20399e-17, 1.103166e-16}} [Gaussian]
{'rotation': 0.025640588253736496, 'scale': {0.013105635, 0.02509005}} [Gaussian]
{'rotation': -6.302142363523444e-09, 'scale': {0.025086053, 0.062259108}} [Gaussian]
{'rotation': -0.024200953543186188, 'scale': {0.011759122, 0.024859667}} [Gaussian]
{'rotation': 2.7553850867612084e-10, 'scale': {5.2863843e-18, 1.1211357e-18}} [Gaussian]
{'rotation': 0.03346157446503639, 'scale': {0.019868067, 0.025513403}} [Gaussian]
